# Gold — LTV por cohort de clientes

Desenvolvido por: Ygor Moraes

## Objetivo

Criar a Gold `gold_ecommerce_clientes_ltv_cohort`, medindo o LTV médio por mês de cadastro dos clientes.

## Regra de negócio

LTV = soma de `valor_total` dos pedidos entregues por cliente.

A cohort é definida pelo mês de cadastro do cliente.

## Decisão de negócio

Considerar apenas pedidos com:

`status_pedido = "ENTREGUE"`

Pedidos cancelados, desconhecidos, enviados ou em processamento não entram no cálculo, pois não representam receita realizada confirmada.

## Fontes

- Silver `ecommerce_clientes`
- Silver `ecommerce_pedidos`

## Cuidados técnicos

- `ecommerce_clientes` é lida como Delta.
- `ecommerce_pedidos` é lida como Delta.
- Pedidos são deduplicados por `id_pedido` antes do cálculo.
- A Gold deve manter uma linha por mês de cohort.

In [0]:
%run "../config/00_config"

In [0]:
%run "../utils/00_utils"

In [0]:
# Importa funções e define parâmetros da Gold.

from pyspark.sql.functions import (
    avg,
    col,
    coalesce,
    count,
    current_timestamp,
    lit,
    month,
    round,
    sum as spark_sum,
    to_date,
    to_timestamp,
    trunc,
    when,
    year,
    row_number
)

from pyspark.sql.window import Window

SILVER_CLIENTES_TABLE = "ecommerce_clientes"
SILVER_PEDIDOS_TABLE = "ecommerce_pedidos"

SILVER_CLIENTES_PATH = f"{SILVER_BASE_PATH}{SILVER_CLIENTES_TABLE}"
SILVER_PEDIDOS_PATH = f"{SILVER_BASE_PATH}{SILVER_PEDIDOS_TABLE}"

GOLD_TABLE = "gold_ecommerce_clientes_ltv_cohort"
GOLD_PATH = f"{GOLD_BASE_PATH}{GOLD_TABLE}"

FINAL_TABLE = f"{TARGET_SCHEMA}.{GOLD_TABLE}"

STATUS_PEDIDO_CONSIDERADO = "ENTREGUE"

CLIENTES_REQUIRED_COLUMNS = [
    "id_cliente",
    "dt_cadastro"
]

PEDIDOS_REQUIRED_COLUMNS = [
    "id_pedido",
    "id_cliente",
    "status_pedido",
    "valor_total",
    "silver_processed_at",
    "dt_ultima_atualizacao_status",
    "dt_pedido"
]

GOLD_KEY_COLUMNS = [
    "ano_cadastro",
    "mes_cadastro",
    "data_cohort"
]

adls_options = get_adls_options()

print("Parâmetros definidos com sucesso.")
print("SILVER_CLIENTES_PATH:", SILVER_CLIENTES_PATH)
print("SILVER_PEDIDOS_PATH:", SILVER_PEDIDOS_PATH)
print("GOLD_PATH:", GOLD_PATH)
print("FINAL_TABLE:", FINAL_TABLE)

In [0]:
# Lê as Silvers necessárias para montar a Gold.

df_clientes = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(SILVER_CLIENTES_PATH)
)

df_pedidos = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(SILVER_PEDIDOS_PATH)
)

total_clientes = df_clientes.count()
total_pedidos = df_pedidos.count()

print("Silver de clientes lida com sucesso.")
print(f"Total clientes: {total_clientes}")

print("Silver de pedidos lida com sucesso em Delta.")
print(f"Total pedidos: {total_pedidos}")

In [0]:
# Valida colunas obrigatórias e chaves principais das fontes.

validate_required_columns(df_clientes, CLIENTES_REQUIRED_COLUMNS)
validate_required_columns(df_pedidos, PEDIDOS_REQUIRED_COLUMNS)

clientes_distintos = (
    df_clientes
    .select(col("id_cliente").cast("int").alias("id_cliente"))
    .distinct()
    .count()
)

pedidos_distintos = (
    df_pedidos
    .select(col("id_pedido").cast("int").alias("id_pedido"))
    .distinct()
    .count()
)

clientes_duplicados = total_clientes - clientes_distintos
pedidos_duplicados = total_pedidos - pedidos_distintos

print(f"Total clientes: {total_clientes}")
print(f"Clientes distintos: {clientes_distintos}")
print(f"Clientes duplicados: {clientes_duplicados}")

print(f"Total pedidos: {total_pedidos}")
print(f"Pedidos distintos: {pedidos_distintos}")
print(f"Pedidos duplicados: {pedidos_duplicados}")

if clientes_duplicados > 0:
    raise Exception("Erro: existem clientes duplicados por id_cliente.")

if pedidos_distintos == 0:
    raise Exception("Erro: não foram encontrados pedidos válidos por id_pedido.")

print("Validação OK: fontes mínimas conferidas.")

In [0]:
# Deduplica pedidos por id_pedido mantendo o registro mais recente.

df_pedidos_base = (
    df_pedidos
    .withColumn("id_pedido_int", col("id_pedido").cast("int"))
    .withColumn("id_cliente_int", col("id_cliente").cast("int"))
    .withColumn("valor_total_decimal", col("valor_total").cast("decimal(18,2)"))
    .withColumn("silver_processed_at_ts", to_timestamp(col("silver_processed_at")))
    .withColumn("dt_ultima_atualizacao_status_ts", to_timestamp(col("dt_ultima_atualizacao_status")))
    .withColumn("dt_pedido_ts", to_timestamp(col("dt_pedido")))
)

window_pedidos_dedup = (
    Window
    .partitionBy("id_pedido_int")
    .orderBy(
        col("silver_processed_at_ts").desc_nulls_last(),
        col("dt_ultima_atualizacao_status_ts").desc_nulls_last(),
        col("dt_pedido_ts").desc_nulls_last()
    )
)

df_pedidos_dedup = (
    df_pedidos_base
    .withColumn("rn", row_number().over(window_pedidos_dedup))
    .filter(col("rn") == 1)
    .drop("rn")
)

print("Pedidos deduplicados por id_pedido.")

In [0]:
# Valida deduplicação e campos necessários para cálculo do LTV.

total_pedidos_dedup = df_pedidos_dedup.count()

pedidos_distintos_dedup = (
    df_pedidos_dedup
    .select("id_pedido_int")
    .distinct()
    .count()
)

pedidos_duplicados_dedup = total_pedidos_dedup - pedidos_distintos_dedup

pedidos_id_nulo = (
    df_pedidos_dedup
    .filter(col("id_pedido_int").isNull())
    .count()
)

clientes_id_nulo_em_pedidos = (
    df_pedidos_dedup
    .filter(col("id_cliente_int").isNull())
    .count()
)

valor_total_nulo = (
    df_pedidos_dedup
    .filter(col("valor_total_decimal").isNull())
    .count()
)

valor_total_negativo = (
    df_pedidos_dedup
    .filter(col("valor_total_decimal") < 0)
    .count()
)

print(f"Total pedidos original: {total_pedidos}")
print(f"Total pedidos após deduplicação: {total_pedidos_dedup}")
print(f"Pedidos distintos após deduplicação: {pedidos_distintos_dedup}")
print(f"Pedidos duplicados restantes: {pedidos_duplicados_dedup}")
print(f"Pedidos com id_pedido nulo: {pedidos_id_nulo}")
print(f"Pedidos com id_cliente nulo: {clientes_id_nulo_em_pedidos}")
print(f"Pedidos com valor_total nulo: {valor_total_nulo}")
print(f"Pedidos com valor_total negativo: {valor_total_negativo}")

if pedidos_duplicados_dedup > 0:
    raise Exception("Erro: ainda existem pedidos duplicados por id_pedido.")

if pedidos_id_nulo > 0:
    raise Exception("Erro: existem pedidos com id_pedido nulo.")

if valor_total_nulo > 0:
    raise Exception("Erro: existem pedidos com valor_total nulo.")

if valor_total_negativo > 0:
    raise Exception("Erro: existem pedidos com valor_total negativo.")

print("Validação OK: pedidos deduplicados e valores conferidos.")

In [0]:
# Considera apenas pedidos ENTREGUE para cálculo de receita realizada.

df_pedidos_entregues = (
    df_pedidos_dedup
    .filter(col("status_pedido") == STATUS_PEDIDO_CONSIDERADO)
)

total_pedidos_entregues = df_pedidos_entregues.count()

clientes_com_pedido_entregue = (
    df_pedidos_entregues
    .filter(col("id_cliente_int").isNotNull())
    .select("id_cliente_int")
    .distinct()
    .count()
)

receita_total_entregue = (
    df_pedidos_entregues
    .agg(spark_sum("valor_total_decimal").alias("receita_total"))
    .collect()[0]["receita_total"]
)

print(f"Pedidos entregues considerados: {total_pedidos_entregues}")
print(f"Clientes com pedido entregue: {clientes_com_pedido_entregue}")
print(f"Receita total entregue: {receita_total_entregue}")

if total_pedidos_entregues == 0:
    raise Exception("Erro: nenhum pedido ENTREGUE encontrado para cálculo do LTV.")

print("Validação OK: pedidos entregues filtrados.")

In [0]:
# Calcula receita total e quantidade de pedidos entregues por cliente.

df_ltv_cliente = (
    df_pedidos_entregues
    .groupBy(col("id_cliente_int").alias("id_cliente"))
    .agg(
        count("id_pedido_int").alias("qtd_pedidos_entregues"),
        spark_sum("valor_total_decimal").alias("receita_total_cliente")
    )
)

total_clientes_ltv = df_ltv_cliente.count()

print(f"Clientes com LTV calculado: {total_clientes_ltv}")
print("LTV por cliente calculado.")

In [0]:
# Cria base de clientes com métricas de LTV.

df_clientes_ltv_base = (
    df_clientes
    .select(
        col("id_cliente").cast("int").alias("id_cliente"),
        col("dt_cadastro")
    )
    .join(
        df_ltv_cliente,
        on="id_cliente",
        how="left"
    )
    .withColumn("qtd_pedidos_entregues", coalesce(col("qtd_pedidos_entregues"), lit(0)))
    .withColumn("receita_total_cliente", coalesce(col("receita_total_cliente"), lit(0).cast("decimal(18,2)")))
    .withColumn(
        "cliente_com_pedido_entregue",
        when(col("qtd_pedidos_entregues") > 0, 1).otherwise(0)
    )
    .withColumn("data_cohort", trunc(to_date(col("dt_cadastro")), "MM"))
    .withColumn("ano_cadastro", year(col("dt_cadastro")))
    .withColumn("mes_cadastro", month(col("dt_cadastro")))
)

print("Base de clientes com LTV criada.")

In [0]:
# Valida se o join manteve um registro por cliente.

total_clientes_ltv_base = df_clientes_ltv_base.count()

clientes_distintos_ltv_base = (
    df_clientes_ltv_base
    .select("id_cliente")
    .distinct()
    .count()
)

clientes_duplicados_ltv_base = total_clientes_ltv_base - clientes_distintos_ltv_base

clientes_sem_cohort = (
    df_clientes_ltv_base
    .filter(col("data_cohort").isNull())
    .count()
)

clientes_com_receita_negativa = (
    df_clientes_ltv_base
    .filter(col("receita_total_cliente") < 0)
    .count()
)

print(f"Total clientes original: {total_clientes}")
print(f"Total clientes na base LTV: {total_clientes_ltv_base}")
print(f"Clientes distintos na base LTV: {clientes_distintos_ltv_base}")
print(f"Clientes duplicados na base LTV: {clientes_duplicados_ltv_base}")
print(f"Clientes sem data_cohort: {clientes_sem_cohort}")
print(f"Clientes com receita negativa: {clientes_com_receita_negativa}")

if total_clientes_ltv_base != total_clientes:
    raise Exception("Erro: a base LTV alterou a quantidade de clientes.")

if clientes_duplicados_ltv_base > 0:
    raise Exception("Erro: existem clientes duplicados na base LTV.")

if clientes_sem_cohort > 0:
    raise Exception("Erro: existem clientes sem data_cohort.")

if clientes_com_receita_negativa > 0:
    raise Exception("Erro: existem clientes com receita negativa.")

print("Validação OK: base LTV manteve 1 registro por cliente.")

In [0]:
# Cria a Gold de LTV por mês de cohort.

df_gold = (
    df_clientes_ltv_base
    .groupBy(
        "ano_cadastro",
        "mes_cadastro",
        "data_cohort"
    )
    .agg(
        count("id_cliente").alias("qtd_clientes_cadastrados"),
        spark_sum("cliente_com_pedido_entregue").alias("qtd_clientes_com_pedido_entregue"),
        spark_sum("qtd_pedidos_entregues").alias("qtd_pedidos_entregues"),
        spark_sum("receita_total_cliente").alias("receita_total_cohort")
    )
    .withColumn(
        "qtd_clientes_sem_pedido_entregue",
        col("qtd_clientes_cadastrados") - col("qtd_clientes_com_pedido_entregue")
    )
    .withColumn(
        "ltv_medio_clientes_cadastrados",
        round(col("receita_total_cohort") / col("qtd_clientes_cadastrados"), 2)
    )
    .withColumn(
        "ltv_medio_clientes_com_pedido_entregue",
        round(
            when(
                col("qtd_clientes_com_pedido_entregue") > 0,
                col("receita_total_cohort") / col("qtd_clientes_com_pedido_entregue")
            ).otherwise(lit(0)),
            2
        )
    )
    .withColumn("status_pedido_considerado", lit(STATUS_PEDIDO_CONSIDERADO))
    .withColumn("gold_processed_at", current_timestamp())
    .orderBy("ano_cadastro", "mes_cadastro")
)

print("Gold de LTV criada em memória.")
display(df_gold)

In [0]:
# Valida totais, cohorts e campos principais da Gold.

total_linhas_gold = df_gold.count()

total_cohorts_distintos = (
    df_gold
    .select(GOLD_KEY_COLUMNS)
    .distinct()
    .count()
)

duplicados_cohort = total_linhas_gold - total_cohorts_distintos

validacao_gold = (
    df_gold
    .agg(
        spark_sum("qtd_clientes_cadastrados").alias("total_clientes_cadastrados"),
        spark_sum("qtd_clientes_com_pedido_entregue").alias("total_clientes_com_pedido_entregue"),
        spark_sum("qtd_clientes_sem_pedido_entregue").alias("total_clientes_sem_pedido_entregue"),
        spark_sum("qtd_pedidos_entregues").alias("total_pedidos_entregues"),
        spark_sum("receita_total_cohort").alias("receita_total")
    )
    .collect()[0]
)

nulos_gold = (
    df_gold
    .filter(
        col("ano_cadastro").isNull() |
        col("mes_cadastro").isNull() |
        col("data_cohort").isNull() |
        col("qtd_clientes_cadastrados").isNull() |
        col("qtd_clientes_com_pedido_entregue").isNull() |
        col("qtd_clientes_sem_pedido_entregue").isNull() |
        col("qtd_pedidos_entregues").isNull() |
        col("receita_total_cohort").isNull() |
        col("ltv_medio_clientes_cadastrados").isNull() |
        col("ltv_medio_clientes_com_pedido_entregue").isNull()
    )
    .count()
)

print(f"Total clientes base: {total_clientes_ltv_base}")
print(f"Total clientes na Gold: {validacao_gold['total_clientes_cadastrados']}")
print(f"Clientes com pedido entregue: {validacao_gold['total_clientes_com_pedido_entregue']}")
print(f"Clientes sem pedido entregue: {validacao_gold['total_clientes_sem_pedido_entregue']}")
print(f"Pedidos entregues na Gold: {validacao_gold['total_pedidos_entregues']}")
print(f"Receita total na Gold: {validacao_gold['receita_total']}")
print(f"Total linhas Gold: {total_linhas_gold}")
print(f"Cohorts duplicados: {duplicados_cohort}")
print(f"Linhas com nulos principais: {nulos_gold}")

if validacao_gold["total_clientes_cadastrados"] != total_clientes_ltv_base:
    raise Exception("Erro: total de clientes da Gold não fecha com a base.")

if (
    validacao_gold["total_clientes_com_pedido_entregue"] +
    validacao_gold["total_clientes_sem_pedido_entregue"]
    != validacao_gold["total_clientes_cadastrados"]
):
    raise Exception("Erro: clientes com + sem pedido entregue não fecha com cadastrados.")

if validacao_gold["total_pedidos_entregues"] != total_pedidos_entregues:
    raise Exception("Erro: total de pedidos entregues da Gold não confere.")

if duplicados_cohort > 0:
    raise Exception("Erro: existem cohorts duplicados na Gold.")

if nulos_gold > 0:
    raise Exception("Erro: existem nulos nas colunas principais da Gold.")

print("Validação OK: Gold em memória conferida.")

In [0]:
# Grava a Gold em Delta no ADLS.

(
    df_gold
    .write
    .format("delta")
    .options(**adls_options)
    .option("overwriteSchema", "true")
    .mode("overwrite")
    .save(GOLD_PATH)
)

print(f"Gold gravada com sucesso em Delta: {GOLD_PATH}")

In [0]:
# Lê e valida a Gold Delta gravada.

df_gold_saved = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(GOLD_PATH)
)

total_linhas_gold_saved = df_gold_saved.count()

total_cohorts_saved = (
    df_gold_saved
    .select(GOLD_KEY_COLUMNS)
    .distinct()
    .count()
)

duplicados_cohort_saved = total_linhas_gold_saved - total_cohorts_saved

validacao_gold_saved = (
    df_gold_saved
    .agg(
        spark_sum("qtd_clientes_cadastrados").alias("total_clientes_cadastrados"),
        spark_sum("qtd_clientes_com_pedido_entregue").alias("total_clientes_com_pedido_entregue"),
        spark_sum("qtd_clientes_sem_pedido_entregue").alias("total_clientes_sem_pedido_entregue"),
        spark_sum("qtd_pedidos_entregues").alias("total_pedidos_entregues"),
        spark_sum("receita_total_cohort").alias("receita_total")
    )
    .collect()[0]
)

print(f"Total linhas Gold Delta: {total_linhas_gold_saved}")
print(f"Cohorts duplicados Gold Delta: {duplicados_cohort_saved}")
print(f"Total clientes base: {total_clientes_ltv_base}")
print(f"Total clientes Gold Delta: {validacao_gold_saved['total_clientes_cadastrados']}")
print(f"Clientes com pedido entregue Gold Delta: {validacao_gold_saved['total_clientes_com_pedido_entregue']}")
print(f"Clientes sem pedido entregue Gold Delta: {validacao_gold_saved['total_clientes_sem_pedido_entregue']}")
print(f"Pedidos entregues Gold Delta: {validacao_gold_saved['total_pedidos_entregues']}")
print(f"Receita total Gold Delta: {validacao_gold_saved['receita_total']}")

if validacao_gold_saved["total_clientes_cadastrados"] != total_clientes_ltv_base:
    raise Exception("Erro: total de clientes da Gold Delta não confere.")

if (
    validacao_gold_saved["total_clientes_com_pedido_entregue"] +
    validacao_gold_saved["total_clientes_sem_pedido_entregue"]
    != validacao_gold_saved["total_clientes_cadastrados"]
):
    raise Exception("Erro: clientes com + sem pedido entregue não fecha na Gold Delta.")

if validacao_gold_saved["total_pedidos_entregues"] != total_pedidos_entregues:
    raise Exception("Erro: total de pedidos entregues da Gold Delta não confere.")

if duplicados_cohort_saved > 0:
    raise Exception("Erro: existem cohorts duplicados na Gold Delta.")

print("Validação OK: Gold Delta gravada corretamente.")

In [0]:
# Prepara a Gold para escrita no SQL Server.

df_gold_sql = (
    df_gold_saved
    .select(
        col("ano_cadastro").cast("int").alias("ano_cadastro"),
        col("mes_cadastro").cast("int").alias("mes_cadastro"),
        col("data_cohort").cast("date").alias("data_cohort"),
        col("qtd_clientes_cadastrados").cast("int").alias("qtd_clientes_cadastrados"),
        col("qtd_clientes_com_pedido_entregue").cast("int").alias("qtd_clientes_com_pedido_entregue"),
        col("qtd_clientes_sem_pedido_entregue").cast("int").alias("qtd_clientes_sem_pedido_entregue"),
        col("qtd_pedidos_entregues").cast("int").alias("qtd_pedidos_entregues"),
        col("receita_total_cohort").cast("decimal(18,2)").alias("receita_total_cohort"),
        col("ltv_medio_clientes_cadastrados").cast("decimal(18,2)").alias("ltv_medio_clientes_cadastrados"),
        col("ltv_medio_clientes_com_pedido_entregue").cast("decimal(18,2)").alias("ltv_medio_clientes_com_pedido_entregue"),
        col("status_pedido_considerado").cast("string").alias("status_pedido_considerado"),
        col("gold_processed_at").cast("timestamp").alias("gold_processed_at")
    )
)

print("Gold preparada para escrita no SQL Server.")
df_gold_sql.printSchema()
display(df_gold_sql.orderBy("ano_cadastro", "mes_cadastro"))

In [0]:
# Grava a Gold diretamente na tabela final do SQL Server.

write_sql_table(
    df=df_gold_sql,
    sql_host=SQL_HOST,
    sql_database=SQL_DATABASE,
    sql_username=SQL_USERNAME,
    sql_password=SQL_PASSWORD,
    table_name=FINAL_TABLE,
    mode="overwrite",
    sql_port=SQL_PORT
)

print(f"Gold gravada com sucesso na tabela final: {FINAL_TABLE}")

In [0]:
# Lê e valida a tabela final do SQL Server.

df_final = read_sql_table(
    spark=spark,
    sql_host=SQL_HOST,
    sql_database=SQL_DATABASE,
    sql_username=SQL_USERNAME,
    sql_password=SQL_PASSWORD,
    table_name=FINAL_TABLE,
    sql_port=SQL_PORT
)

total_linhas_final = df_final.count()

total_cohorts_final = (
    df_final
    .select(GOLD_KEY_COLUMNS)
    .distinct()
    .count()
)

cohorts_duplicados_final = total_linhas_final - total_cohorts_final

validacao_final = (
    df_final
    .agg(
        spark_sum("qtd_clientes_cadastrados").alias("total_clientes_cadastrados"),
        spark_sum("qtd_clientes_com_pedido_entregue").alias("total_clientes_com_pedido_entregue"),
        spark_sum("qtd_clientes_sem_pedido_entregue").alias("total_clientes_sem_pedido_entregue"),
        spark_sum("qtd_pedidos_entregues").alias("total_pedidos_entregues"),
        spark_sum("receita_total_cohort").alias("receita_total")
    )
    .collect()[0]
)

print(f"Total linhas tabela final: {total_linhas_final}")
print(f"Cohorts duplicados tabela final: {cohorts_duplicados_final}")
print(f"Total clientes base: {total_clientes_ltv_base}")
print(f"Total clientes tabela final: {validacao_final['total_clientes_cadastrados']}")
print(f"Clientes com pedido entregue tabela final: {validacao_final['total_clientes_com_pedido_entregue']}")
print(f"Clientes sem pedido entregue tabela final: {validacao_final['total_clientes_sem_pedido_entregue']}")
print(f"Pedidos entregues tabela final: {validacao_final['total_pedidos_entregues']}")
print(f"Receita total tabela final: {validacao_final['receita_total']}")

if validacao_final["total_clientes_cadastrados"] != total_clientes_ltv_base:
    raise Exception("Erro: total de clientes da tabela final não confere.")

if (
    validacao_final["total_clientes_com_pedido_entregue"] +
    validacao_final["total_clientes_sem_pedido_entregue"]
    != validacao_final["total_clientes_cadastrados"]
):
    raise Exception("Erro: clientes com + sem pedido entregue não fecha na tabela final.")

if validacao_final["total_pedidos_entregues"] != total_pedidos_entregues:
    raise Exception("Erro: total de pedidos entregues da tabela final não confere.")

if cohorts_duplicados_final > 0:
    raise Exception("Erro: existem cohorts duplicados na tabela final.")

print("Validação OK: tabela final SQL Server gravada corretamente.")